# Part 1k (Extended) — Video & Document Image Augmentation
**Deep Learning Assignment — Part 1k remaining modalities**

| Section | Modality | Dataset / Source |
|---------|----------|-----------------|
| 1k-vi | **Video** | UCF-101 via `tensorflow_datasets` (frame simulation) |
| 1k-vii | **Document Images** | RVL-CDIP style synthetic scanned document |

These complete the full Part 1k augmentation coverage:
image ✓ · video ✓ · text ✓ · time-series ✓ · tabular ✓ · speech ✓ · document images ✓

## Cell 0 — Install & Imports

In [ ]:
!pip install opencv-python-headless pillow numpy matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image, ImageFilter, ImageEnhance
import io, warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("Imports OK")

## 1k-vi — Video Data Augmentation

**What is a video tensor?**  
Shape `(T, H, W, C)` — T frames, each an H×W×C image.

**Real datasets for video classification:**
- **UCF-101** — 101 action categories, ~13k clips (`tensorflow_datasets` in Colab)
- **Kinetics-400** — 400 classes, large scale
- **HMDB-51** — 51 action categories

**Key insight:** Video augmentations must be applied *consistently across all frames*  
(same crop, same flip) to preserve temporal coherence. Applying different transforms  
to different frames creates unrealistic artifacts.

We simulate a video as a `(16, 64, 64, 3)` tensor representing 16 RGB frames.

In [ ]:
# ── Simulate a real-looking video (or load from UCF-101 in Colab) ─────────────
# In Colab with internet: use tensorflow_datasets
#   import tensorflow_datasets as tfds
#   ds = tfds.load('ucf101', split='train[:1%]')
#   video = next(iter(ds))['video'].numpy()  # shape: (T, H, W, 3)
#
# Here we create a synthetic video: smooth temporal gradient + spatial noise
T, H, W, C = 16, 64, 64, 3

# Frame 0 is red-tinted, last frame is blue-tinted — simulates motion over time
video = np.zeros((T, H, W, C), dtype=np.float32)
for t in range(T):
    progress = t / (T - 1)
    # Background gradient changes over time
    video[t, :, :, 0] = 1.0 - progress * 0.5     # R channel fades
    video[t, :, :, 2] = progress * 0.8            # B channel rises
    video[t, :, :, 1] = 0.3 + 0.2 * np.sin(progress * np.pi)
    # Add spatial structure — moving "object"
    cx = int(W * 0.2 + progress * W * 0.6)        # object moves left→right
    cy = H // 2
    rr, cc = np.ogrid[:H, :W]
    mask = (rr-cy)**2 + (cc-cx)**2 < 8**2
    video[t, mask] = [0.9, 0.8, 0.1]              # yellow circle
    # Add a little noise
    video[t] += np.random.normal(0, 0.02, (H, W, C))
video = np.clip(video, 0, 1)

print(f"Simulated video: {video.shape}  (T={T} frames, {H}x{W}, {C} channels)")
print(f"Pixel range: [{video.min():.3f}, {video.max():.3f}]")

# ── Visualise original: show every 4th frame ───────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for i, ax in enumerate(axes):
    ax.imshow(video[i*4])
    ax.set_title(f'Frame {i*4}', fontsize=9)
    ax.axis('off')
plt.suptitle('Original Simulated Video (every 4th frame)')
plt.tight_layout(); plt.show()

In [ ]:
# ══ Video Augmentation Functions ═══════════════════════════════════════════════

def video_temporal_crop(v, n_frames=12):
    """
    Randomly crop n_frames consecutive frames.
    Reduces temporal length — simulates starting/stopping the clip at different points.
    Key: crop is RANDOM but consistent — same start frame for the whole clip.
    """
    start = np.random.randint(0, len(v) - n_frames + 1)
    return v[start:start + n_frames]


def video_horizontal_flip(v):
    """
    Flip all frames horizontally.
    Applied UNIFORMLY — all frames flipped identically, preserving temporal coherence.
    """
    return v[:, :, ::-1, :]   # flip W axis


def video_brightness_jitter(v, delta=0.2):
    """
    Random brightness shift applied uniformly to all frames.
    Same delta for every frame — avoids unrealistic flickering.
    """
    shift = np.random.uniform(-delta, delta)
    return np.clip(v + shift, 0, 1)


def video_temporal_reverse(v):
    """
    Reverse the frame order — video plays backwards.
    Useful for actions with temporal symmetry (e.g. jumping up / jumping down).
    """
    return v[::-1].copy()


def video_frame_drop(v, drop_prob=0.15):
    """
    Randomly drop frames (replace with previous frame).
    Simulates packet loss or variable frame rate acquisition.
    """
    out = v.copy()
    for t in range(1, len(v)):
        if np.random.rand() < drop_prob:
            out[t] = out[t-1]   # repeat previous frame
    return out


def video_spatial_crop(v, crop_ratio=0.8):
    """
    Random spatial crop applied identically to ALL frames, then resize.
    Simulates zoom / reframing while preserving motion patterns.
    """
    crop_h = int(H * crop_ratio)
    crop_w = int(W * crop_ratio)
    top    = np.random.randint(0, H - crop_h + 1)
    left   = np.random.randint(0, W - crop_w + 1)
    # Same crop coordinates for all frames — temporal consistency
    cropped = v[:, top:top+crop_h, left:left+crop_w, :]
    # Resize back to original dimensions
    resized = np.stack([
        cv2.resize(frame, (W, H), interpolation=cv2.INTER_LINEAR)
        for frame in cropped
    ])
    return resized


def video_gaussian_noise(v, sigma=0.04):
    """
    Add independent Gaussian noise to each frame.
    Unlike spatial/temporal augmentations, noise is frame-independent.
    Simulates sensor noise in low-light conditions.
    """
    return np.clip(v + np.random.normal(0, sigma, v.shape), 0, 1)


def video_color_jitter(v, hue_shift=0.05):
    """
    Shift hue uniformly across all frames.
    Applied consistently — simulates different lighting conditions.
    """
    shift = np.random.uniform(-hue_shift, hue_shift)
    return np.clip(v + np.array([shift, 0, -shift]), 0, 1)


# ── Apply all augmentations ────────────────────────────────────────────────────
np.random.seed(42)
augmentations = {
    'Original':          video,
    'Temporal Crop (12f)': video_temporal_crop(video, 12),
    'Horizontal Flip':   video_horizontal_flip(video),
    'Brightness +0.2':   video_brightness_jitter(video, 0.2),
    'Temporal Reverse':  video_temporal_reverse(video),
    'Frame Drop (15%)':  video_frame_drop(video, 0.15),
    'Spatial Crop (80%)':video_spatial_crop(video, 0.8),
    'Gaussian Noise':    video_gaussian_noise(video, 0.05),
}

print("Augmented video shapes:")
for name, v in augmentations.items():
    print(f"  {name:25s}: {v.shape}  [{v.min():.2f}, {v.max():.2f}]")

In [ ]:
# ── Visualise: show frame 8 for each augmentation ─────────────────────────────
# Frame 8 is the midpoint — best shows temporal augmentation effects

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (name, v) in zip(axes.flat, augmentations.items()):
    frame_idx = min(8, len(v) - 1)
    ax.imshow(np.clip(v[frame_idx], 0, 1))
    ax.set_title(f'{name}\n(frame {frame_idx})', fontsize=8)
    ax.axis('off')
plt.suptitle('1k-vi — Video Augmentation (showing frame 8 of each)', fontsize=12)
plt.tight_layout(); plt.show()

# ── Temporal consistency check: show all frames of one augmentation ────────────
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
flip_v = video_horizontal_flip(video)
for t in range(16):
    row, col = t // 8, t % 8
    axes[row, col].imshow(np.clip(flip_v[t], 0, 1))
    axes[row, col].set_title(f'f{t}', fontsize=7)
    axes[row, col].axis('off')
plt.suptitle('Horizontal Flip — All 16 Frames (consistent spatial transform)', fontsize=11)
plt.tight_layout(); plt.show()

print("\n── Why each augmentation matters ──────────────────────────────────────")
print("Temporal Crop:    model must classify from partial clips — length invariance")
print("Horizontal Flip:  doubles training data for symmetric actions")
print("Brightness:       robust to different lighting conditions")
print("Temporal Reverse: learns bidirectional action patterns")
print("Frame Drop:       robust to variable frame rates / packet loss")
print("Spatial Crop:     scale/position invariance")
print("Gaussian Noise:   robust to sensor noise")

## 1k-vii — Document Image Augmentation

**Why document image augmentation?**  
Document AI models (OCR, form understanding, invoice parsing) are sensitive to:
- Scanner quality and resolution
- Physical page condition (folds, stains, skew)  
- Lighting and contrast variation
- Compression artifacts from scanning

**Real datasets:**
- **RVL-CDIP** — 400k document images, 16 categories (forms, letters, memos, etc.)
- **FUNSD** — form understanding dataset
- **DocVQA** — document visual question answering

**Key difference from natural image augmentation:**  
Aggressive color/hue changes are less useful (documents are mostly grayscale text).  
Instead: perspective distortion, blur, noise, compression, and rotation matter most.

In [ ]:
# ── Create a realistic synthetic document image ───────────────────────────────
# Simulates a scanned page with text lines, a title, and a table

DOC_H, DOC_W = 400, 300

def create_synthetic_document():
    """Generate a synthetic document image resembling a scanned form/letter."""
    img = np.ones((DOC_H, DOC_W, 3), dtype=np.uint8) * 245   # off-white paper

    # Title bar
    img[20:35, 30:270] = 30    # dark title text block

    # Text lines (simulating paragraphs)
    for row in range(50, 200, 14):
        width = np.random.randint(150, 240)
        img[row:row+5, 30:30+width] = np.random.randint(20, 60)

    # Table structure
    # Horizontal lines
    for row in [220, 240, 260, 280, 300, 320]:
        img[row:row+2, 30:270] = 80
    # Vertical lines
    for col in [30, 110, 190, 270]:
        img[220:322, col:col+2] = 80
    # Table cells content
    for row in range(242, 320, 20):
        for col_start in [35, 115, 195]:
            w = np.random.randint(20, 60)
            img[row:row+5, col_start:col_start+w] = np.random.randint(30, 80)

    # Page margin lines
    img[10:390, 15:17] = 180
    img[10:390, 283:285] = 180

    return img

doc = create_synthetic_document()
print(f"Synthetic document: {doc.shape}  dtype={doc.dtype}")
print(f"Pixel range: [{doc.min()}, {doc.max()}]")

plt.figure(figsize=(4, 5))
plt.imshow(doc, cmap='gray' if doc.ndim==2 else None)
plt.title('Synthetic Document (scanned form)', fontsize=10)
plt.axis('off'); plt.tight_layout(); plt.show()

In [ ]:
# ══ Document Image Augmentation Functions ══════════════════════════════════════

def doc_perspective_warp(img, strength=0.04):
    """
    Perspective transformation — simulates page curl or scanner tilt.
    The four corners are randomly displaced, then the image is warped.
    Key for robustness: documents are rarely perfectly flat on a scanner.
    """
    h, w = img.shape[:2]
    src = np.float32([[0,0],[w,0],[w,h],[0,h]])
    noise = np.random.uniform(-strength, strength, (4, 2)) * np.array([w, h])
    dst = np.clip(src + noise.astype(np.float32), 0, [w,h])
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, M, (w, h),
                               borderMode=cv2.BORDER_CONSTANT,
                               borderValue=(245,245,245))


def doc_gaussian_blur(img, max_sigma=1.5):
    """
    Gaussian blur — simulates out-of-focus scanning or low DPI.
    Small blur (sigma < 0.5): imperceptible.
    Medium blur (sigma 1–1.5): noticeably softer text edges.
    """
    sigma = np.random.uniform(0.3, max_sigma)
    pil   = Image.fromarray(img)
    return np.array(pil.filter(ImageFilter.GaussianBlur(radius=sigma)))


def doc_brightness_contrast(img, b_range=(0.75, 1.25), c_range=(0.7, 1.3)):
    """
    Random brightness and contrast adjustment.
    Simulates uneven illumination, faded documents, or overexposed scans.
    """
    pil = Image.fromarray(img)
    pil = ImageEnhance.Brightness(pil).enhance(np.random.uniform(*b_range))
    pil = ImageEnhance.Contrast(pil).enhance(np.random.uniform(*c_range))
    return np.array(pil)


def doc_salt_pepper(img, amount=0.002):
    """
    Salt-and-pepper noise — simulates scanner dust, speckles, ink bleed.
    Salt = white pixels (scanner reflection)
    Pepper = black pixels (dirt, toner specks)
    """
    out = img.copy()
    n_pixels = int(amount * img.shape[0] * img.shape[1])
    # Salt (white)
    ys = np.random.randint(0, img.shape[0], n_pixels)
    xs = np.random.randint(0, img.shape[1], n_pixels)
    out[ys, xs] = 255
    # Pepper (black)
    ys2 = np.random.randint(0, img.shape[0], n_pixels)
    xs2 = np.random.randint(0, img.shape[1], n_pixels)
    out[ys2, xs2] = 0
    return out


def doc_rotation(img, max_deg=3.0):
    """
    Small random rotation — simulates imperfect document placement on scanner.
    Kept small (≤3°) since larger angles are unrealistic for scanned documents.
    """
    angle = np.random.uniform(-max_deg, max_deg)
    pil   = Image.fromarray(img)
    return np.array(pil.rotate(angle,
                               resample=Image.BICUBIC,
                               fillcolor=(245,245,245) if img.ndim==3 else 245))


def doc_jpeg_compression(img, quality=None):
    """
    JPEG compression artifacts — simulates low-quality scanning or file storage.
    Quality 70–100: minimal artifacts.
    Quality 30–60: noticeable blocking, especially around text edges.
    Quality < 30: severe degradation.
    """
    if quality is None:
        quality = np.random.randint(25, 65)
    pil = Image.fromarray(img)
    buf = io.BytesIO()
    pil.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return np.array(Image.open(buf))


def doc_shadow(img, intensity=0.3):
    """
    Gradient shadow — simulates uneven lighting across the page
    (e.g. shadow from book spine when scanning a book page).
    """
    h, w = img.shape[:2]
    shadow = np.linspace(1.0, 1.0 - intensity, w)   # left to right darkening
    out = img.astype(np.float32) * shadow[np.newaxis, :, np.newaxis]
    return np.clip(out, 0, 255).astype(np.uint8)


# ── Apply all augmentations ────────────────────────────────────────────────────
np.random.seed(42)
doc_augmentations = [
    ('Original',              doc),
    ('Perspective Warp',      doc_perspective_warp(doc, 0.04)),
    ('Gaussian Blur',         doc_gaussian_blur(doc, 1.5)),
    ('Brightness/Contrast',   doc_brightness_contrast(doc)),
    ('Salt & Pepper',         doc_salt_pepper(doc, 0.003)),
    ('Rotation (±3°)',        doc_rotation(doc, 3.0)),
    ('JPEG Compression q=35', doc_jpeg_compression(doc, quality=35)),
    ('Shadow Gradient',       doc_shadow(doc, 0.35)),
]

print("Document augmentation shapes:")
for name, aug in doc_augmentations:
    print(f"  {name:28s}: {aug.shape}  range=[{aug.min()},{aug.max()}]")

In [ ]:
# ── Visualise all augmentations ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, (name, aug) in zip(axes.flat, doc_augmentations):
    ax.imshow(aug, cmap='gray' if aug.ndim==2 else None, vmin=0, vmax=255)
    ax.set_title(name, fontsize=8)
    ax.axis('off')
plt.suptitle('1k-vii — Document Image Augmentation\n'
             '(Simulated scanned form: perspective, blur, noise, compression, shadow)',
             fontsize=11)
plt.tight_layout(); plt.show()

# ── Show zoomed comparison on text region ─────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
zoom_region = (40, 110, 30, 200)   # rows 40:110, cols 30:200
y1,y2,x1,x2 = zoom_region
selected = [('Original', doc),
            ('Blur σ=1.5', doc_gaussian_blur(doc,1.5)),
            ('JPEG q=35',  doc_jpeg_compression(doc,35)),
            ('Salt+Pepper', doc_salt_pepper(doc,0.004))]
for ax, (name, aug) in zip(axes, selected):
    ax.imshow(aug[y1:y2, x1:x2], cmap='gray', vmin=0, vmax=255)
    ax.set_title(name, fontsize=9); ax.axis('off')
plt.suptitle('Zoomed text region — effect of augmentations on text readability')
plt.tight_layout(); plt.show()

# ── Pipeline: apply multiple augmentations in sequence ────────────────────────
def doc_augmentation_pipeline(img, p=0.5):
    """
    Stochastic augmentation pipeline — each transform applied with probability p.
    Used during training: each document gets a different random combination.
    """
    if np.random.rand() < p: img = doc_rotation(img, max_deg=2.0)
    if np.random.rand() < p: img = doc_perspective_warp(img, strength=0.03)
    if np.random.rand() < p: img = doc_brightness_contrast(img)
    if np.random.rand() < p: img = doc_gaussian_blur(img, max_sigma=1.0)
    if np.random.rand() < p: img = doc_salt_pepper(img, amount=0.001)
    if np.random.rand() < p: img = doc_jpeg_compression(img)
    if np.random.rand() < p: img = doc_shadow(img, intensity=0.2)
    return img

fig, axes = plt.subplots(1, 6, figsize=(20, 4))
axes[0].imshow(doc, vmin=0, vmax=255); axes[0].set_title('Original'); axes[0].axis('off')
for i, ax in enumerate(axes[1:], 1):
    aug = doc_augmentation_pipeline(doc.copy(), p=0.6)
    ax.imshow(aug, vmin=0, vmax=255)
    ax.set_title(f'Pipeline {i}', fontsize=8); ax.axis('off')
plt.suptitle('Stochastic Augmentation Pipeline — 5 different random augmentations of same document')
plt.tight_layout(); plt.show()
print("✓ Document image augmentation complete")